# Deep Recurrent Neural Networks

:label:`sec_deep_rnn`

Up until now, we have focused on defining networks 
consisting of a sequence input, 
a single hidden RNN layer,
and an output layer. 
Despite having just one hidden layer 
between the input at any time step
and the corresponding output,
there is a sense in which these networks are deep.
Inputs from the first time step can influence
the outputs at the final time step $T$ 
(often 100s or 1000s of steps later).
These inputs pass through $T$ applications
of the recurrent layer before reaching 
the final output. 
However, we often also wish to retain the ability
to express complex relationships 
between the inputs at a given time step
and the outputs at that same time step.
Thus we often construct RNNs that are deep
not only in the time direction 
but also in the input-to-output direction.
This is precisely the notion of depth
that we have already encountered 
in our development of MLPs
and deep CNNs.


The standard method for building this sort of deep RNN 
is strikingly simple: we stack the RNNs on top of each other. 
Given a sequence of length $T$, the first RNN produces 
a sequence of outputs, also of length $T$.
These, in turn, constitute the inputs to the next RNN layer. 
In this short section, we illustrate this design pattern
and present a simple example for how to code up such stacked RNNs.
Below, in :numref:`fig_deep_rnn`, we illustrate
a deep RNN with $L$ hidden layers.
Each hidden state operates on a sequential input
and produces a sequential output.
Moreover, any RNN cell (white box in :numref:`fig_deep_rnn`) at each time step
depends on both the same layer's 
value at the previous time step
and the previous layer's value 
at the same time step. 

![Architecture of a deep RNN.](../img/deep-rnn.svg)
:label:`fig_deep_rnn`

Formally, suppose that we have a minibatch input
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$ 
(number of examples $=n$; number of inputs in each example $=d$) at time step $t$.
At the same time step, 
let the hidden state of the $l^\textrm{th}$ hidden layer ($l=1,\ldots,L$) be $\mathbf{H}_t^{(l)} \in \mathbb{R}^{n \times h}$ 
(number of hidden units $=h$)
and the output layer variable be 
$\mathbf{O}_t \in \mathbb{R}^{n \times q}$ 
(number of outputs: $q$).
Setting $\mathbf{H}_t^{(0)} = \mathbf{X}_t$,
the hidden state of
the $l^\textrm{th}$ hidden layer
that uses the activation function $\phi_l$
is calculated as follows:

$$\mathbf{H}_t^{(l)} = \phi_l(\mathbf{H}_t^{(l-1)} \mathbf{W}_{\textrm{xh}}^{(l)} + \mathbf{H}_{t-1}^{(l)} \mathbf{W}_{\textrm{hh}}^{(l)}  + \mathbf{b}_\textrm{h}^{(l)}),$$
:eqlabel:`eq_deep_rnn_H`

where the weights $\mathbf{W}_{\textrm{xh}}^{(l)} \in \mathbb{R}^{h \times h}$ and $\mathbf{W}_{\textrm{hh}}^{(l)} \in \mathbb{R}^{h \times h}$, together with
the bias $\mathbf{b}_\textrm{h}^{(l)} \in \mathbb{R}^{1 \times h}$, 
are the model parameters of the $l^\textrm{th}$ hidden layer.

At the end, the calculation of the output layer 
is only based on the hidden state 
of the final $L^\textrm{th}$ hidden layer:

$$\mathbf{O}_t = \mathbf{H}_t^{(L)} \mathbf{W}_{\textrm{hq}} + \mathbf{b}_\textrm{q},$$

where the weight $\mathbf{W}_{\textrm{hq}} \in \mathbb{R}^{h \times q}$ 
and the bias $\mathbf{b}_\textrm{q} \in \mathbb{R}^{1 \times q}$ 
are the model parameters of the output layer.

Just as with MLPs, the number of hidden layers $L$ 
and the number of hidden units $h$ are hyperparameters
that we can tune.
Common RNN layer widths ($h$) are in the range $(64, 2056)$,
and common depths ($L$) are in the range $(1, 8)$. 
In addition, we can easily get a deep-gated RNN
by replacing the hidden state computation in :eqref:`eq_deep_rnn_H`
with that from an LSTM or a GRU.


In [1]:
import torch
from torch import nn

In [2]:
%matplotlib inline

import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

## Implementation from Scratch

To implement a multilayer RNN from scratch,
we can treat each layer as an `RNNScratch` instance
with its own learnable parameters.


In [3]:
from utils.helper import Module, RNNScratch

class StackedRNNScratch(Module):
    def __init__(self, num_inputs, num_hiddens, num_layers, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.rnns = nn.Sequential(*[RNNScratch(
            num_inputs if i==0 else num_hiddens, num_hiddens, sigma)
                                    for i in range(num_layers)])

The multilayer forward computation
simply performs forward computation
layer by layer.


In [4]:
from utils.helper import add_to_class

@add_to_class(StackedRNNScratch)
def forward(self, inputs, Hs=None):
    outputs = inputs
    if Hs is None: Hs = [None] * self.num_layers
    for i in range(self.num_layers):
        outputs, Hs[i] = self.rnns[i](outputs, Hs[i])
        outputs = torch.stack(outputs, 0)
    return outputs, Hs

As an example, we train a deep GRU model on
*The Time Machine* dataset (same as in :numref:`sec_rnn-scratch`).
To keep things simple we set the number of layers to 2.


In [5]:
import inspect
print("StackedRNNScratch.forward:", inspect.signature(StackedRNNScratch.forward))
print("RNNScratch.forward:", inspect.signature(RNNScratch.forward))

StackedRNNScratch.forward: (self, inputs, Hs=None)
RNNScratch.forward: (self, X)


In [6]:
from utils.helper import StackedRNNScratch, TimeMachine, RNNLMScratch, Trainer

data = TimeMachine(batch_size=1024, num_steps=32)
rnn_block = StackedRNNScratch(num_inputs=len(data.vocab),
                              num_hiddens=32, num_layers=2)
model = RNNLMScratch(rnn_block, vocab_size=len(data.vocab), lr=2)
trainer = Trainer(max_epochs=10, gradient_clip_val=1)
trainer.fit(model, data)

epoch 1, train loss 3.099819, val loss 2.942604
epoch 2, train loss 2.924598, val loss 2.880554
epoch 3, train loss 2.887453, val loss 2.857794
epoch 4, train loss 2.872076, val loss 2.846530
epoch 5, train loss 2.863924, val loss 2.839797
epoch 6, train loss 2.858666, val loss 2.835291
epoch 7, train loss 2.855153, val loss 2.832140
epoch 8, train loss 2.852542, val loss 2.829710
epoch 9, train loss 2.850611, val loss 2.827933
epoch 10, train loss 2.849104, val loss 2.826405


## Concise Implementation


Fortunately many of the logistical details required
to implement multiple layers of an RNN 
are readily available in high-level APIs.
Our concise implementation will use such built-in functionalities.
The code generalizes the one we used previously in :numref:`sec_gru`,
letting us specify the number of layers explicitly 
rather than picking the default of only one layer.


In [7]:

from utils.helper import Module, RNN

class GRU(RNN):  
    """The multilayer GRU model."""
    def __init__(self, num_inputs, num_hiddens, num_layers, dropout=0):
        Module.__init__(self)
        self.save_hyperparameters()
        self.rnn = nn.GRU(num_inputs, num_hiddens, num_layers,
                          dropout=dropout)

The architectural decisions such as choosing hyperparameters 
are very similar to those of :numref:`sec_gru`.
We pick the same number of inputs and outputs 
as we have distinct tokens, i.e., `vocab_size`.
The number of hidden units is still 32.
The only difference is that we now 
(**select a nontrivial number of hidden layers 
by specifying the value of `num_layers`.**)


In [8]:
from utils.helper import RNNLM

gru = GRU(num_inputs=len(data.vocab), num_hiddens=32, num_layers=2)
model = RNNLM(gru, vocab_size=len(data.vocab), lr=2)
trainer.fit(model, data)

epoch 1, train loss 2.978729, val loss 2.981329
epoch 2, train loss 2.877562, val loss 2.864433
epoch 3, train loss 2.859445, val loss 2.832132
epoch 4, train loss 2.847934, val loss 2.823019
epoch 5, train loss 2.842248, val loss 2.821201
epoch 6, train loss 2.835621, val loss 2.809918
epoch 7, train loss 2.824803, val loss 2.799591
epoch 8, train loss 2.797188, val loss 2.775031
epoch 9, train loss 2.765362, val loss 2.700050
epoch 10, train loss 2.719135, val loss 2.652641


In [9]:
model.predict('it has', 20, data.vocab)

'it has ae the the the the '

## Summary

In deep RNNs, the hidden state information is passed 
to the next time step of the current layer 
and the current time step of the next layer.
There exist many different flavors of deep RNNs, such as LSTMs, GRUs, or vanilla RNNs. 
Conveniently, these models are all available 
as parts of the high-level APIs of deep learning frameworks.
Initialization of models requires care. 
Overall, deep RNNs require considerable amount of work 
(such as learning rate and clipping) 
to ensure proper convergence.
